# V017: pre-outcome secondary confirmation of fixed postprocessing

This is a separate, preregistered follow-on study. It leaves V014's source, split, weights, epochs, fits and original decision unchanged. The additional pressure-grid/weighted-median policy was selected on A and independently audited before E outcomes.

Reusing **cold E-fit** predictions is valid here only if all source/timing gates pass. Both policy and this consumer's reviewed source must have been frozen before the first last-member (`deep_curve`) E fit began; V014 scores only after all six fits. Otherwise reject this E reuse and require another confirmation split. Independent primary E audit is mandatory, and a failed primary E cannot be rescued. Fit the pressure grid anew from E-fit labels only. Test exactly the selected policy against the fixed six-model mean and five-model baseline; no alternative/weight/threshold search.

This uses another grouped split relative to A; it is resampling evidence, not a pristine test. No modeling fits or submissions occur here. The primary workflow remains independently governed by its existing frozen protocol. Default execution is inert.


In [ ]:
import os,json,hashlib
from pathlib import Path
from datetime import datetime,timezone
import numpy as np
import pandas as pd
V017_PROTOCOL_SHA="97214e460ea200a836922245363132c09ef54b133080b89bb342c4e1d191d627"

def v017_sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
def v017_ah(a):return hashlib.sha256(np.ascontiguousarray(a).tobytes()).hexdigest()
def v017_sig(a):return hashlib.sha256(json.dumps(a,sort_keys=True,separators=(",",":"),allow_nan=False).encode()).hexdigest()
def v017_code(p):
    return "\n".join("".join(c["source"]) for c in json.loads(Path(p).read_text())["cells"] if c["cell_type"]=="code")
def v017_require(c,m):
    if not bool(c):raise ValueError(m)
def v017_timing(predeclared_utc,source_frozen_utc,first_last_fit_utc,completed_utc):
    def t(s):return datetime.fromisoformat(s.replace("Z","+00:00"))
    return max(t(predeclared_utc),t(source_frozen_utc))<t(first_last_fit_utc)<t(completed_utc)
def v017_interval(new_error,old_error,counts,seed):
    sums=np.add.reduceat(new_error-old_error,np.r_[0,np.cumsum(counts)[:-1]])
    rng=np.random.default_rng(seed);draws=[]
    for _ in range(10):
        take=rng.integers(0,len(counts),size=(100,len(counts)));draws.extend(sums[take].sum(1)/counts[take].sum(1))
    lo,hi=np.quantile(draws,[.025,.975])
    return dict(gain=float(old_error.mean()-new_error.mean()),difference=float((new_error-old_error).mean()),low=float(lo),high=float(hi),seed=seed,draws=1000,unit="whole breath",point_weighted=True)

def run_postprocessing_confirmation_v017(root=".",expected_primary_audit_sha256=None,expected_source_code_sha256=None):
    root=Path(root).resolve();out=root/"outputs/postprocessing_confirmation_v017";E=root/"outputs/confirmation_v014"
    source=hashlib.sha256(v017_code(root/"research/postprocessing_confirmation_v017.ipynb").encode()).hexdigest()
    v017_require(source==expected_source_code_sha256,"Secondary source differs from reviewed pin")
    proto=root/"research/postprocessing_confirmation_protocol_v017.json"
    v017_require(v017_sha(proto)==V017_PROTOCOL_SHA,"Secondary preregistration changed")
    plan=json.loads(proto.read_text())
    for p,h in plan["input_hashes"].items():v017_require(v017_sha(root/p)==h,"Preregistered input changed: "+p)
    frozen=json.loads((root/"research/postprocessing_confirmation_preflight_v017.json").read_text())
    v017_require(frozen["status"]=="passed" and frozen["source_code_sha256"]==source and frozen["protocol_sha256"]==V017_PROTOCOL_SHA,"Secondary source preflight changed")
    result_path=E/"confirmation_result.json";result=json.loads(result_path.read_text());lock=json.loads((E/"locked_before_E.json").read_text());split=json.loads((E/"E_split.json").read_text())
    auditpath=root/"research/confirmation_review_v014.json"
    v017_require(expected_primary_audit_sha256 is not None and v017_sha(auditpath)==expected_primary_audit_sha256,"Independent primary E audit pin required")
    audit=json.loads(auditpath.read_text())
    v017_require(audit["status"]=="passed" and audit["confirmation_result_sha256"]==v017_sha(result_path) and audit["selection_sha256"]==result["selection_sha256"],"Primary audit does not bind E result")
    v017_require(result["status"]=="complete" and result["source_code_sha256"]==plan["E_source_code_sha256"] and result["protocol_sha256"]==plan["E_protocol_sha256"] and result["winner_weights"]==plan["weights"],"Primary E recipe/source changed")
    v017_require(result["selection_sha256"]==v017_sig(lock) and result["locked_file_sha256"]==v017_sha(E/"locked_before_E.json") and lock["E_split_file_sha256"]==v017_sha(E/"E_split.json"),"E lock/split changed")
    events=[json.loads(line) for line in (E/"events.jsonl").read_text().splitlines() if line.strip()]
    starts=[e["time_utc"] for e in events if e["event"]=="cold_E_fit_started" and e.get("member")=="deep_curve"]
    v017_require(starts and v017_timing(plan["created_utc"],frozen["created_utc"],min(starts),result["completed_utc"]),"Secondary was not frozen before the last E member began; require another confirmation split")
    v017_require(result["passed"] is True,"Primary E failed; secondary cannot rescue it")
    components=root/"research/postprocessing_components_v016.ipynb";code=v017_code(components)
    v017_require(hashlib.sha256(code.encode()).hexdigest()==plan["component_code_sha256"],"Fixed postprocessing changed")
    ns={};exec(compile(code,components.name,"exec"),ns)
    v017_require(v017_sha(root/"data/train.csv")=="2dc30dc5c8e603c8c68200830eab7ed8c7af42c823f8ae68b0beb663eea817d7","Approved train changed")
    frame=pd.read_csv(root/"data/train.csv",usecols=["id","breath_id","u_out","pressure"],dtype=dict(id="int64",breath_id="int32",u_out="int8",pressure="float32"))
    bids=frame.breath_id.to_numpy()[::80];y=frame.pressure.to_numpy().reshape(-1,80);phase=frame.u_out.to_numpy().reshape(-1,80);ids=frame.id.to_numpy().reshape(-1,80)
    fit=np.asarray(split["indices"]["E_fit"],np.int64);val=np.asarray(split["indices"]["E_validation"],np.int64);aval=np.asarray(split["indices"]["A_validation"],np.int64)
    v017_require(len(fit)==54323 and len(val)==6791 and not np.intersect1d(fit,val).size and not np.intersect1d(np.r_[fit,val],aval).size,"E grouping changed")
    mask=phase[val]==0;counts=mask.sum(1);truth=y[val][mask].astype(np.float64);rows=ids[val][mask]
    v017_require(v017_ah(truth)==result["E_truth_array_sha256"] and v017_ah(rows)==result["E_row_ids_array_sha256"] and np.array_equal(rows,np.load(E/"E_row_ids.npy")),"E native truth/row alignment changed")
    codebook=ns["pressure_codebook_v016"](y[fit],phase[fit])
    predictions={}
    for name,a in result["prediction_artifacts"].items():
        v017_require(v017_sha(root/a["file"])==a["file_sha256"],"Cold E prediction file changed")
        p=np.load(root/a["file"],allow_pickle=False)
        v017_require(v017_ah(p)==a["array_sha256"] and a["row_ids_sha256"]==v017_ah(rows) and p.shape==truth.shape and p.dtype==np.float64,"Cold E prediction alignment changed")
        predictions[name]=p
    selected=ns["selected_postprocessing_v016"](predictions,plan["weights"],codebook)
    mean=sum(w*predictions[k] for k,w in result["winner_weights"].items())
    baseline=sum(w*predictions[k] for k,w in result["baseline_weights"].items())
    v017_require(np.array_equal(mean,np.load(E/"candidate_E_predictions.npy")) and np.array_equal(baseline,np.load(E/"baseline_E_predictions.npy")),"Primary fixed blends changed")
    comparisons={name:v017_interval(np.abs(selected-truth),np.abs(ref-truth),counts,plan["bootstrap_seed"]) for name,ref in [("six_member_mean",mean),("five_member_baseline",baseline)]}
    passed=bool(all(c["gain"]>0 and c["high"]<0 for c in comparisons.values()))
    out.mkdir(parents=True,exist_ok=True)
    v017_require(not (out/"confirmation_result.json").exists(),"Completed secondary result is preserved")
    np.save(out/"E_fit_pressure_codebook.npy",codebook);np.save(out/"selected_E_predictions.npy",selected)
    report=dict(status="complete",passed=passed,policy=plan["selected_policy"],protocol_sha256=V017_PROTOCOL_SHA,source_code_sha256=source,component_code_sha256=plan["component_code_sha256"],primary_result_sha256=v017_sha(result_path),primary_audit_sha256=expected_primary_audit_sha256,selection_sha256=result["selection_sha256"],timing_gate_passed=True,preregistered_utc=plan["created_utc"],source_frozen_utc=frozen["created_utc"],first_last_E_fit_utc=min(starts),primary_E_completed_utc=result["completed_utc"],native_mean_MAE=float(np.abs(selected-truth).mean()),median_absolute_error=float(np.median(np.abs(selected-truth))),comparisons=comparisons,scored_points=len(rows),training_grid_fit_breaths=len(fit),training_grid_levels=len(codebook),training_grid_file_sha256=v017_sha(out/"E_fit_pressure_codebook.npy"),prediction_file_sha256=v017_sha(out/"selected_E_predictions.npy"),prediction_array_sha256=v017_ah(selected),row_ids_array_sha256=v017_ah(rows),no_weight_or_threshold_fit_on_E=True,primary_pipeline_unchanged=True,no_new_model_fits=True,submission_performed=False,official_best_preserved=.18025,confirmation_is_resampling_evidence=True,completed_utc=datetime.now(timezone.utc).isoformat())
    (out/"confirmation_result.json").write_text(json.dumps(report,indent=2)+"\n")
    return report

if os.environ.get("PA3_RUN_POSTPROCESS_V017","0")=="1":
    V017_RESULT=run_postprocessing_confirmation_v017(Path.cwd(),expected_primary_audit_sha256=os.environ.get("PA3_V017_PRIMARY_AUDIT_SHA256"),expected_source_code_sha256=os.environ.get("PA3_V017_SOURCE_SHA256"))
    print(json.dumps(V017_RESULT,indent=2))
